In [1]:
import os

REPO_URL = "https://github.com/sinh2206/O_D.git"
REPO_DIR = "O_D"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch --all --prune
    !git checkout -B $(git symbolic-ref --short refs/remotes/origin/HEAD | sed 's@^origin/@@')
    !git reset --hard origin/$(git symbolic-ref --short refs/remotes/origin/HEAD | sed 's@^origin/@@')
    %cd ..

%cd {REPO_DIR}
!git log -1 --oneline


Cloning into 'O_D'...
remote: Enumerating objects: 9802, done.
remote: Counting objects: 100% (235/235), done.
remote: Compressing objects: 100% (165/165), done.
remote: Total 9802 (delta 169), reused 130 (delta 69), pack-reused 9567 (from 2)
Receiving objects: 100% (9802/9802), 1010.14 MiB | 55.41 MiB/s, done.
Resolving deltas: 100% (435/435), done.
Updating files: 100% (9025/9025), done.
/kaggle/working/O_D
1621b06 (HEAD -> main, origin/main, origin/HEAD) 14/6


In [2]:
!pwd
!ls


/kaggle/working/O_D
img_error.py  models	       predict.py  requirements.txt  utils
LICENSE       note.txt	       public	   small_obj.py
main.tex      obj-detec.ipynb  README.md   train.py


In [3]:
%pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 102.9 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 req

In [4]:
!python train.py \
  --train_data ./public/annotations/train.json \
  --val_data ./public/annotations/val.json \
  --image_dir ./public/train/images \
  --val_image_dir ./public/val/images \
  --checkpoint_dir ./models/

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100%|███████████████████████████████████████| 83.3M/83.3M [00:00<00:00, 193MB/s]
Device: cuda, AMP: True
Train samples: 7500, Val samples: 1500, Classes: ['person', 'car', 'dog', 'cat', 'chair']
Balanced sampling: True
Class weights enabled: True
Num workers: requested=2, resolved=2, max_safe=4
Pin memory: True
Class weights: [0.45, 0.9275, 1.1709, 1.4433, 1.249]
Epoch 001/030 | train_loss=1.6900 (cls=0.5572, reg=0.6589, ctr=0.6760) | val_loss=1.3289
Saved best checkpoint: models/best.pth (val_loss=1.3289)
Epoch 002/030 | train_loss=1.2766 (cls=0.3967, reg=0.4850, ctr=0.6623) | val_loss=1.1511
Saved best checkpoint: models/best.pth (val_loss=1.1511)
Epoch 003/030 | train_loss=1.1810 (cls=0.3745, reg=0.4304, ctr=0.6547) | val_loss=1.1014
Saved best checkpoint: models/best.pth (val_loss=1.1014)
Epoch 004/030 | train_loss=1.0820 (cls=0.3341, reg=0.3916, ctr=

In [5]:
!python predict.py \
  --image_dir ./public/val/images \
  --output val_predictions.json \
  --model_path ./models/best.pth

Device: cuda
Checkpoint meta: epoch=27, best_val_loss=0.8060917695786091, img_size=640
Predicted images: 1500
Saved JSON: val_predictions.json
TTA fallback: False (min_votes=2)
Hardcase items: 50
Hardcase summary: results/hardcase_summary.json


In [6]:
!python public/tools/evaluate_predictions.py \
  --ground_truth public/annotations/val.json \
  --predictions val_predictions.json \
  --output val_score.json

{
  "mAP@0.5": 0.732613,
  "performance_points": 20,
  "iou_threshold": 0.5,
  "num_ground_truth_boxes": 2021,
  "num_predictions": 3372,
  "micro_precision": 0.476572,
  "micro_recall": 0.795151,
  "per_class": {
    "person": {
      "ap": 0.780899,
      "num_ground_truth": 1074,
      "num_predictions": 1318,
      "true_positives": 870,
      "false_positives": 448,
      "recall": 0.810056,
      "precision": 0.660091
    },
    "car": {
      "ap": 0.697453,
      "num_ground_truth": 283,
      "num_predictions": 529,
      "true_positives": 213,
      "false_positives": 316,
      "recall": 0.75265,
      "precision": 0.402647
    },
    "dog": {
      "ap": 0.776923,
      "num_ground_truth": 206,
      "num_predictions": 383,
      "true_positives": 172,
      "false_positives": 211,
      "recall": 0.834951,
      "precision": 0.449086
    },
    "cat": {
      "ap": 0.835127,
      "num_ground_truth": 176,
      "num_predictions": 269,
      "true_positives": 152,
      "fa

In [7]:
# Zip project excluding public
import os
from pathlib import Path

src = Path('/kaggle/working/O_D')
out_zip = Path('/kaggle/working/train.zip')

if not src.exists():
    raise FileNotFoundError(f'Not found: {src}')

cmd = f"cd {src} && zip -r {out_zip} . -x 'public/*' 'public/**'"
print(cmd)
ret = os.system(cmd)
if ret != 0:
    raise RuntimeError(f'zip failed with code {ret}')
print(f'Created: {out_zip}')


cd /kaggle/working/O_D && zip -r /kaggle/working/train.zip . -x 'public/*' 'public/**'
  adding: .git/ (stored 0%)
  adding: .git/config (deflated 31%)
  adding: .git/logs/ (stored 0%)
  adding: .git/logs/refs/ (stored 0%)
  adding: .git/logs/refs/heads/ (stored 0%)
  adding: .git/logs/refs/heads/main (deflated 27%)
  adding: .git/logs/refs/remotes/ (stored 0%)
  adding: .git/logs/refs/remotes/origin/ (stored 0%)
  adding: .git/logs/refs/remotes/origin/HEAD (deflated 27%)
  adding: .git/logs/HEAD (deflated 27%)
  adding: .git/objects/ (stored 0%)
  adding: .git/objects/info/ (stored 0%)
  adding: .git/objects/pack/ (stored 0%)
  adding: .git/objects/pack/pack-8bd4e604ebc82af39ad0a9447b7bd19c4b32599e.idx (deflated 0%)
  adding: .git/objects/pack/pack-8bd4e604ebc82af39ad0a9447b7bd19c4b32599e.pack (deflated 0%)
  adding: .git/info/ (stored 0%)
  adding: .git/info/exclude (deflated 28%)
  adding: .git/index (deflated 61%)
  adding: .git/branches/ (stored 0%)
  adding: .git/refs/ (stored 0%